In [1]:
%pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# Sistema de Alertas Automáticas — CX Intelligence
## Acción 1: Detección y priorización de clientes insatisfechos

**Analista:** Rodolfo Gabriel Riveros Lobos
**Fecha:** 29/05/2026
**Herramienta:** DistilBERT (HuggingFace) + exportación Excel

---

### Contexto operativo

El equipo de CX recibe diariamente nuevas reseñas de clientes.
Este sistema las clasifica automáticamente, asigna nivel de prioridad
y genera un reporte listo para que cada agente sepa a quién contactar primero.

**Input:** reseñas nuevas del día
**Output:** archivo Excel con alertas priorizadas y acción recomendada

## Imports y Setup

In [10]:
import pandas as pd
import numpy as np
import os
import kagglehub
from transformers import pipeline
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from datetime import date
import warnings

warnings.filterwarnings('ignore')

print("✅ Imports correctos")
print(f"Fecha del reporte: {date.today()}")

✅ Imports correctos
Fecha del reporte: 2026-05-29


## Carga y simulación de reseñas nuevas

In [11]:
# Carga del dataset
path = kagglehub.dataset_download("nicapotato/womens-ecommerce-clothing-reviews")
df = pd.read_csv(
    os.path.join(path, "Womens Clothing E-Commerce Reviews.csv"),
    index_col=0
)

# Simulamos que las últimas 500 reseñas son las "nuevas del día"
# En producción real este CSV vendría de la plataforma de reseñas
df_new = df[['Review Text', 'Rating', 'Department Name']].dropna().tail(500).copy()
df_new.columns = ['review', 'rating', 'department']
df_new = df_new.reset_index(drop=True)

print(f"Reseñas nuevas del día: {len(df_new)}")
print(f"\nDistribución por departamento:")
print(df_new['department'].value_counts())
print(f"\nMuestra de reseñas:")
df_new[['review', 'rating']].head(3)

Reseñas nuevas del día: 500

Distribución por departamento:
department
Dresses     182
Tops        157
Bottoms      96
Intimate     43
Jackets      19
Trend         3
Name: count, dtype: int64

Muestra de reseñas:


,review,rating
0,I love the way this dress fits and flows! it d...,5
1,I ordered this sweater in xxsp and the sweater...,3
2,I ordered an xxs petite and this hangs like a ...,2


## SECCIÓN 3 — CLASIFICACIÓN AUTOMÁTICA CON DISTILBERT

In [ ]:


# Cargamos el modelo (primera vez descarga ~250MB)
print("Cargando modelo... ⏳")
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
print("✅ Modelo listo")

# Preparar textos truncados a 512 caracteres
texts = df_new['review'].apply(lambda x: str(x)[:512]).tolist()

print(f"Clasificando {len(texts)} reseñas... ⏳")
results = classifier(texts, truncation=True, max_length=512, batch_size=16)
print("✅ Clasificación completada")

# Agregar resultados al dataframe
df_new['hf_label'] = [r['label'] for r in results]
df_new['hf_score'] = [round(r['score'], 4) for r in results]

# Mapear etiquetas
df_new['sentiment'] = df_new['hf_label'].map({
    'NEGATIVE': 'Negative',
    'POSITIVE': 'Positive'
})

print(f"\nDistribución de sentimientos detectados:")
print(df_new['sentiment'].value_counts())

Cargando modelo... ⏳


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Modelo listo
Clasificando 500 reseñas... ⏳
✅ Clasificación completada

Distribución de sentimientos detectados:
sentiment
Positive    355
Negative    145
Name: count, dtype: int64


## SECCIÓN 4 — SISTEMA DE PRIORIDAD EN TRES NIVELES
# ============================================================
# Lógica de priorización para el equipo CX:
#
# CRÍTICO  → Negativo con score > 0.95  (certeza muy alta)
# ALTO     → Negativo con score 0.80–0.95
# MEDIO    → Negativo con score < 0.80
# (los positivos no generan alerta)
# ============================================================

In [ ]:


def assign_priority(row):
    if row['sentiment'] == 'Negative':
        if row['hf_score'] >= 0.95:
            return 'CRÍTICO'
        elif row['hf_score'] >= 0.80:
            return 'ALTO'
        else:
            return 'MEDIO'
    else:
        return 'OK'

def assign_action(row):
    actions = {
        'CRÍTICO': 'Contactar cliente en menos de 2 horas',
        'ALTO':    'Contactar cliente antes del cierre del día',
        'MEDIO':   'Revisar reseña y evaluar contacto',
        'OK':      'Sin acción requerida'
    }
    return actions[row['priority']]

df_new['priority'] = df_new.apply(assign_priority, axis=1)
df_new['action']   = df_new.apply(assign_action, axis=1)

# Resumen ejecutivo
print("=== RESUMEN DE ALERTAS DEL DÍA ===\n")
summary = df_new['priority'].value_counts()
for level in ['CRÍTICO', 'ALTO', 'MEDIO', 'OK']:
    count = summary.get(level, 0)
    print(f"  {level:<10} → {count:>4} reseñas")

print(f"\nTotal reseñas procesadas: {len(df_new)}")
print(f"Total alertas generadas:  "
      f"{len(df_new[df_new['priority'] != 'OK'])}")

=== RESUMEN DE ALERTAS DEL DÍA ===

  CRÍTICO    →  107 reseñas
  ALTO       →   25 reseñas
  MEDIO      →   13 reseñas
  OK         →  355 reseñas

Total reseñas procesadas: 500
Total alertas generadas:  145


## Construcción del reporte

In [ ]:


# Filtramos solo las alertas (excluimos OK para el reporte operativo)
df_alerts = df_new[df_new['priority'] != 'OK'].copy()

# Ordenamos por prioridad y score
priority_order = {'CRÍTICO': 0, 'ALTO': 1, 'MEDIO': 2}
df_alerts['priority_order'] = df_alerts['priority'].map(priority_order)
df_alerts = df_alerts.sort_values(
    ['priority_order', 'hf_score'],
    ascending=[True, False]
).drop('priority_order', axis=1)

# Reporte final con columnas para el equipo CX
df_report = df_alerts[[
    'priority', 'action', 'department',
    'review', 'rating', 'hf_score', 'sentiment'
]].copy()

df_report.columns = [
    'Prioridad', 'Acción recomendada', 'Departamento',
    'Reseña', 'Rating original', 'Score modelo', 'Sentimiento'
]

df_report = df_report.reset_index(drop=True)
df_report.index += 1  # índice desde 1 para el equipo

print(f"Alertas en el reporte: {len(df_report)}")
print(f"\nPrimeras 5 alertas:")
df_report.head()

Alertas en el reporte: 145

Primeras 5 alertas:


,Prioridad,Acción recomendada,Departamento,Reseña,Rating original,Score modelo,Sentimiento
1,CRÍTICO,Contactar cliente en menos de 2 horas,Tops,I had high hopes ordering this top...unfortuna...,1,0.9997,Negative
2,CRÍTICO,Contactar cliente en menos de 2 horas,Intimate,Oh what a disappointment! i was looking forwar...,2,0.9997,Negative
3,CRÍTICO,Contactar cliente en menos de 2 horas,Tops,"This top looked so cute in the picture, but un...",2,0.9997,Negative
4,CRÍTICO,Contactar cliente en menos de 2 horas,Dresses,"Unfortunately, this dress did not work for me ...",2,0.9997,Negative
5,CRÍTICO,Contactar cliente en menos de 2 horas,Bottoms,Lots of pairs of the pilcro serif-- and for so...,3,0.9996,Negative


## Exportación a Excel con formato

In [ ]:

output_path = '../outputs/alerts_report_29052026.xlsx'

# Exportar base
df_report.to_excel(output_path, index=True, sheet_name='Alertas CX')

# Aplicar formato con openpyxl
wb = load_workbook(output_path)
ws = wb['Alertas CX']

# Colores por prioridad
colors = {
    'CRÍTICO': 'FF4444',  # rojo
    'ALTO':    'FF9900',  # naranja
    'MEDIO':   'FFDD44',  # amarillo
}

# Formato del encabezado
header_fill = PatternFill(
    start_color='1F3864',
    end_color='1F3864',
    fill_type='solid'
)
header_font = Font(color='FFFFFF', bold=True)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center')

# Formato de filas por prioridad
for row in ws.iter_rows(min_row=2):
    priority_val = row[1].value  # columna Prioridad
    if priority_val in colors:
        fill = PatternFill(
            start_color=colors[priority_val],
            end_color=colors[priority_val],
            fill_type='solid'
        )
        for cell in row:
            cell.fill = fill

# Ancho de columnas
column_widths = {
    'A': 8,   # índice
    'B': 12,  # prioridad
    'C': 38,  # acción
    'D': 18,  # departamento
    'E': 60,  # reseña
    'F': 14,  # rating
    'G': 14,  # score
    'H': 14,  # sentimiento
}
for col, width in column_widths.items():
    ws.column_dimensions[col].width = width

wb.save(output_path)
print(f"✅ Reporte guardado en: {output_path}")
print(f"📊 {len(df_report)} alertas exportadas")

✅ Reporte guardado en: ../outputs/alerts_report_29052026.xlsx
📊 145 alertas exportadas


---

## Resultado operativo

El sistema procesó 500 reseñas nuevas del día en menos de 2 minutos
y generó un reporte priorizado listo para el equipo de CX.

### Qué recibe el equipo cada día

| Prioridad | Criterio | Acción |
|---|---|---|
| CRÍTICO | Score negatividad > 95% | Contactar en menos de 2 horas |
| ALTO | Score negatividad 80–95% | Contactar antes del cierre |
| MEDIO | Score negatividad < 80% | Revisar y evaluar |
| OK | Reseña positiva | Sin acción requerida |

### Valor operativo

El equipo de CX recibe cada mañana un Excel con las alertas del día
ordenadas por urgencia. Sin leer una sola reseña manualmente.
Sin criterio subjetivo. Sin clientes insatisfechos que se pierdan
entre el volumen.